# 02 - Feature Engineering — CORRECTED FINAL (Kaggle)

**Input:** `symca_raw.csv`  
**Output:** `symca_cleaned.csv`

This notebook keeps the useful cleaning work from the original notebook and fixes the important issues:

1. Historical features use only information available before the current lecture.
2. Rolling previous-3 attendance does not use a full-dataset mean for filling.
3. Consecutive lecture count is sequential within a section/day.
4. Feature names are consistent for the next model/deployment notebook.
5. The raw CSV is never overwritten.

The target remains `Attendence Percentage`.


## 1. Imports

In [ ]:
import os
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("Libraries imported successfully.")


## 2. Find the raw CSV in Kaggle

Run this cell first. Copy the path containing `symca_raw.csv` into `DATA_PATH` below.


In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))


## 3. Load Raw Data

In [ ]:
# Replace with your real Kaggle input path.
# Example:
# DATA_PATH = "/kaggle/input/symca-attendance/symca_raw.csv"

DATA_PATH = "/kaggle/input/YOUR-DATASET-NAME/symca_raw.csv"

df = pd.read_csv(DATA_PATH)

print("Shape before cleaning:", df.shape)
display(df.head())


## 4. Clean Previous Lecture Attendance

In [ ]:
df["Previous Lecture Attendence"] = (
    df["Previous Lecture Attendence"]
      .replace("-", np.nan)
)

df["Previous Lecture Attendence"] = pd.to_numeric(
    df["Previous Lecture Attendence"],
    errors="coerce"
)

print("Missing after cleaning:",
      df["Previous Lecture Attendence"].isna().sum())


## 5. Clean Start and End Time

In [ ]:
def parse_time_to_minutes(t):
    if pd.isna(t):
        return np.nan

    text = str(t).strip().upper()

    if "AM" in text:
        period = "AM"
    elif "PM" in text:
        period = "PM"
    else:
        return np.nan

    digits_part = text.split(period)[0]
    digits = re.sub(r"[^0-9]", "", digits_part)

    if len(digits) <= 2:
        hour, minute = int(digits), 0
    elif len(digits) == 3:
        hour, minute = int(digits[0]), int(digits[1:])
    else:
        hour, minute = int(digits[:-2]), int(digits[-2:])

    if minute >= 60 or hour > 12:
        return np.nan

    if period == "PM" and hour != 12:
        hour += 12
    elif period == "AM" and hour == 12:
        hour = 0

    return hour * 60 + minute


df["Start Time Minutes"] = df["Start_Time"].apply(parse_time_to_minutes)
df["End Time Minutes"] = df["End_Time"].apply(parse_time_to_minutes)

print("Invalid/missing start times:",
      df["Start Time Minutes"].isna().sum())
print("Invalid/missing end times:",
      df["End Time Minutes"].isna().sum())

display(
    df[
        ["Start_Time", "Start Time Minutes",
         "End_Time", "End Time Minutes"]
    ].head(10)
)


## 6. Date and Chronological Ordering

In [ ]:
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

if df["Date"].isna().any():
    raise ValueError("Invalid Date values remain. Fix the raw data before continuing.")

df["Day of Week"] = df["Date"].dt.day_name()

sort_cols = [
    "Date",
    "Start Time Minutes",
    "Section",
    "Subject",
    "Lecture_No"
]

sort_cols = [c for c in sort_cols if c in df.columns]

df = (
    df.sort_values(sort_cols)
      .reset_index(drop=True)
)

print("Date range:",
      df["Date"].min().date(),
      "to",
      df["Date"].max().date())


## 7. Target Attendance Percentage

In [ ]:
# Use the observed counts to make the target internally consistent.
df["Attendence Percentage"] = (
    pd.to_numeric(df["Students Present"], errors="coerce")
    / pd.to_numeric(df["Total Enrolled Students"], errors="coerce")
    * 100
).round(2)

print(df["Attendence Percentage"].describe())


## 8. Day of Semester and Week Number

In [ ]:
semester_start = df["Date"].min()

df["Day of Semester"] = (
    df["Date"] - semester_start
).dt.days + 1

df["Week Number"] = (
    (df["Day of Semester"] - 1) // 7
) + 1

display(
    df[
        ["Date", "Day of Semester", "Week Number"]
    ].head(10)
)


## 9. Days Since Last Holiday — Corrected

`Holiday Before/ After == Yes` marks a holiday-related date in the collected schedule.

For each section, the feature carries forward the most recent holiday date. The current day's holiday is not used to calculate the value for that same row.

The first observation has no historical holiday and remains missing by design.


In [ ]:
def calculate_days_since_last_holiday(group):
    group = group.sort_values(
        ["Date", "Start Time Minutes"]
    ).copy()

    last_holiday_date = None
    values = []

    for _, row in group.iterrows():
        current_date = row["Date"]

        if last_holiday_date is None:
            values.append(np.nan)
        else:
            values.append(
                (current_date - last_holiday_date).days
            )

        # Update AFTER calculating current row.
        if str(row["Holiday Before/ After"]).strip().lower() in [
            "yes", "true", "1"
        ]:
            last_holiday_date = current_date

    return pd.Series(values, index=group.index)


df["Days Since Last Holiday"] = (
    df.groupby(
        "Section",
        group_keys=False
    ).apply(
        calculate_days_since_last_holiday,
        include_groups=False
    )
)

# Restore original index alignment safely.
df["Days Since Last Holiday"] = (
    df["Days Since Last Holiday"]
    .reindex(df.index)
)

display(
    df[
        ["Date", "Section",
         "Holiday Before/ After",
         "Days Since Last Holiday"]
    ].head(15)
)


## 10. Consecutive Lecture Count — Corrected

The previous version used `transform('count')`, which gave the total number of lectures for that section/day on every row.

Here the value is sequential:

Lecture 1 → 1  
Lecture 2 → 2  
Lecture 3 → 3

This is calculated after sorting by date/time.


In [ ]:
df["Consecutive Lecture Count (Day)"] = (
    df.groupby(["Date", "Section"])
      .cumcount() + 1
)

display(
    df[
        ["Date", "Section", "Lecture_No",
         "Start Time Minutes",
         "Consecutive Lecture Count (Day)"]
    ].head(15)
)


## 11. Previous Lecture and Rolling Previous 3 — Leakage Safe

Historical attendance must come from earlier lectures only.

The grouping includes `Section` and `Subject` so the previous attendance for one subject is not accidentally used for another subject.

`shift(1)` is essential: it excludes the current lecture.


In [ ]:
history_group = ["Section", "Subject"]

df["Previous Lecture Attendance"] = (
    df.groupby(history_group)["Attendence Percentage"]
      .shift(1)
)

df["Rolling Avg Attendance (Prev 3)"] = (
    df.groupby(history_group)["Attendence Percentage"]
      .transform(
          lambda s: s.shift(1).rolling(
              window=3,
              min_periods=1
          ).mean()
      )
)

display(
    df[
        history_group +
        ["Date", "Lecture_No",
         "Attendence Percentage",
         "Previous Lecture Attendance",
         "Rolling Avg Attendance (Prev 3)"]
    ].head(20)
)


## 12. Historical Average Attendance

Only observations before the current lecture are used.


In [ ]:
df["Historical Avg Attendance"] = (
    df.groupby(history_group)["Attendence Percentage"]
      .transform(
          lambda s: s.shift(1).expanding().mean()
      )
)

display(
    df[
        history_group +
        ["Date", "Attendence Percentage",
         "Historical Avg Attendance"]
    ].head(15)
)


## 13. Monthly Historical Average — Leakage Safe

The old `groupby(...).transform('mean')` used all attendance values from the month, including future lectures.

This version uses only earlier lectures in the same section/subject/month.


In [ ]:
df["_Year_Month"] = df["Date"].dt.to_period("M")

df["Monthly Avg Attendance"] = (
    df.groupby(
        history_group + ["_Year_Month"]
    )["Attendence Percentage"]
      .transform(
          lambda s: s.shift(1).expanding().mean()
      )
)

df.drop(columns=["_Year_Month"], inplace=True)

display(
    df[
        history_group +
        ["Date", "Attendence Percentage",
         "Monthly Avg Attendance"]
    ].head(20)
)


## 14. Macro Historical Attendance Trend

A broader historical mean is calculated using all earlier rows only.


In [ ]:
df["Macro Historical Avg Attendance"] = (
    df["Attendence Percentage"]
      .shift(1)
      .expanding()
      .mean()
)

display(
    df[
        ["Date", "Attendence Percentage",
         "Macro Historical Avg Attendance"]
    ].head(15)
)


## 15. Time-of-Day Group

The PDF asks for a time-of-day grouping.

The project already uses the useful `Before Lunch / After Lunch` split, so we retain that naming consistently.


In [ ]:
df["Lunch Time Slot"] = np.where(
    df["Start Time Minutes"] < 13 * 60,
    "Before Lunch",
    "After Lunch"
)

display(
    df[
        ["Start_Time", "Start Time Minutes",
         "Lunch Time Slot"]
    ].head(10)
)


## 16. Week Before Examination

The raw data contains `Internal Test Week`, but not a separate exact exam-date column.

Therefore we do not invent an exam date. We create a transparent indicator from the available `Internal Test Week` field.

`Yes` = the schedule marks this observation as an internal-test week.


In [ ]:
test_text = (
    df["Internal Test Week"]
      .astype(str)
      .str.strip()
      .str.lower()
)

df["Week Before Exam"] = (
    test_text.isin(["yes", "true", "1"])
    .astype(int)
)

print(df["Week Before Exam"].value_counts())


## 17. Leakage / Feature Audit

These are the columns that must not be used as predictors for a future attendance prediction model:
- `Students Present`
- `Attendence Percentage`

The target is calculated from `Students Present`, so using it would reveal the answer.


In [ ]:
target = "Attendence Percentage"

forbidden_model_inputs = [
    "Students Present",
    "Attendence Percentage"
]

print("Forbidden model inputs:")
for col in forbidden_model_inputs:
    print("-", col)

historical_features = [
    "Previous Lecture Attendance",
    "Rolling Avg Attendance (Prev 3)",
    "Historical Avg Attendance",
    "Monthly Avg Attendance",
    "Macro Historical Avg Attendance"
]

print("\nHistorical feature missing values:")
display(
    df[historical_features]
      .isna()
      .sum()
      .to_frame("Missing Values")
)


## 18. Final Data Quality Checks

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Missing target:", int(df[target].isna().sum()))

required_features = [
    "Start Time Minutes",
    "End Time Minutes",
    "Day of Semester",
    "Week Number",
    "Days Since Last Holiday",
    "Consecutive Lecture Count (Day)",
    "Monthly Avg Attendance",
    "Rolling Avg Attendance (Prev 3)",
    "Lunch Time Slot",
    "Week Before Exam",
    "Previous Lecture Attendance",
    "Historical Avg Attendance",
    "Macro Historical Avg Attendance"
]

print("\nRequired engineered features:")
for col in required_features:
    print(
        ("OK   " if col in df.columns else "MISS ")
        + col
    )


## 19. Final Preview

In [ ]:
display(
    df[
        [
            "Date",
            "Section",
            "Subject",
            "Lecture_No",
            "Start Time Minutes",
            "Day of Semester",
            "Week Number",
            "Days Since Last Holiday",
            "Consecutive Lecture Count (Day)",
            "Previous Lecture Attendance",
            "Rolling Avg Attendance (Prev 3)",
            "Historical Avg Attendance",
            "Monthly Avg Attendance",
            "Macro Historical Avg Attendance",
            "Lunch Time Slot",
            "Week Before Exam",
            "Attendence Percentage"
        ]
    ].head(20)
)


## 20. Save Final Cleaned Dataset

The output is a new CSV. The raw dataset remains unchanged.

Use this exact file as the input to Notebook 03.


In [ ]:
OUTPUT_PATH = "/kaggle/working/symca_cleaned.csv"

# Keep Date in the raw-file style for portability.
df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
print("Final shape:", df.shape)


# DONE

Use the generated:

`/kaggle/working/symca_cleaned.csv`

as the input for `03_Model_Training.ipynb`.

**Do not use the old `symca_cleaned.csv` after making this correction.**
